<a href="https://colab.research.google.com/github/am-3/IB9AU-2026/blob/main/Task16_Intelligent_Financial_Agent_with_Gemini_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 16: Intelligent Financial Agent with Gemini Tools
Atharva M
___
## Objective
This notebook demonstrates the creation and refinement of an intelligent financial agent capable of retrieving real-time stock information and performing basic financial analysis using Google's Gemini-2.5-Flash model. The primary goal is to empower the agent with custom tools to fetch critical financial metrics like stock prices, risk scores (Beta), and Price-to-Earnings (P/E) ratios, enabling it to respond to user queries with data-driven insights. This specific task focuses on implementing a P/E ratio tool and using it to assess stock valuation.
___
## Tech Stack
*   **Google Gemini-2.5-Flash**: The large language model (LLM) serving as the core of the intelligent agent, chosen for its speed and advanced tool-use capabilities.
*   **`google-generativeai` SDK**: Python client library for interacting with the Gemini API.
*   **`yfinance`**: A powerful and popular Python library for fetching financial market data from Yahoo Finance.
*   **Python 3**: The programming language used for development.
*   **Google Colab**: The cloud-based Jupyter notebook environment for development and execution.
___
## Methodology
1.  **Environment Setup**: Initialize the Colab environment, install necessary libraries (`google-generativeai`, `yfinance`), and configure API access using securely stored API keys.
2.  **Tool Definition**: Implement Python functions (`get_stock_price`, `get_company_risk_score`, `get_pe_ratio`) that encapsulate external API calls to `yfinance` to retrieve specific financial data. These functions are designed to be "callable" by the Gemini model.
3.  **Agent Initialization**: Instantiate the `gemini-2.5-flash` model and provide it with the defined tools. Automatic function calling is enabled to allow the model to autonomously decide when and how to use these tools to fulfill user requests.
4.  **Query and Analysis**: The agent processes natural language queries, intelligently selecting and executing the appropriate financial tools. It then synthesizes the retrieved data to provide analytical responses, such as evaluating stock valuation based on P/E ratios.
5.  **Code Optimization and Narrative**: The notebook code is optimized for PEP 8 compliance, enhanced with clear comments, and structured to provide a professional, coherent narrative suitable for a technical portfolio.
___

In [ ]:
# 1. Setup (Use the standard library)
# Get your API key from https://aistudio.google.com/
!pip install -q -U google-generativeai # Ensure the correct library is installed
import google.generativeai as genai
from google.colab import userdata
import os

# Configure the Gemini API with the securely stored API key.
# The API key is fetched from Colab's secrets manager.
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# Install the yfinance library for fetching stock data.
!pip install -q yfinance

In [ ]:
import yfinance as yf

# Define the "Real" Tools for fetching financial data

def get_stock_price(ticker: str):
    """
    Retrieves the current live stock price for a given ticker.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'NVDA').
    """
    print(f"  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for {ticker} ...")

    try:
        stock = yf.Ticker(ticker)
        # Using .fast_info for quicker retrieval of basic price information.
        price = stock.fast_info['last_price']
        return round(price, 2)
    except Exception as e:
        return f"Error fetching price for {ticker}: {e}"


def get_company_risk_score(ticker: str):
    """
    Calculates a risk proxy based on the stock's Beta (market volatility).
    A Beta > 1.0 means higher risk/volatility than the market.
    Args:
        ticker: The stock ticker symbol.
    """
    print(f"  ... ⚠️ TOOL CALL: Fetching Risk Metrics for {ticker} ...")

    try:
        stock = yf.Ticker(ticker)
        # Fetch Beta from stock's info, default to 0 if not available.
        beta = stock.info.get('beta', 0)

        # Categorize risk for human-readable assessment.
        if beta > 1.5:
            assessment = "High Risk (High Volatility)"
        elif beta < 0.8:
            assessment = "Low Risk (Stable)"
        else:
            assessment = "Moderate Risk"

        return {"beta": beta, "assessment": assessment}
    except Exception as e:
        return "Risk data unavailable"

print("✅ Real-Time Tools Defined.")

✅ Real-Time Tools Defined.


In [ ]:
# 4. Initialize Model with Tools
# We use 'gemini-2.5-flash' as it is fast and excellent at tool use.

tools_list = [get_stock_price, get_company_risk_score]

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    tools=tools_list
)

# Enable automatic function calling.
# This tells the SDK: "If the model asks to run a function, just run it for me and give the answer back."
chat = model.start_chat(enable_automatic_function_calling=True)

print("✅ Model initialized with Agentic capabilities.")

✅ Model initialized with Agentic capabilities.


In [ ]:
# 5. Test: Single Tool Call to retrieve Apple's stock price.
response = chat.send_message("What is the current stock price of Apple?")

print("\n🤖 AGENT RESPONSE:")
print(response.text)

  ... 🔍 TOOL CALL: Connecting to Yahoo Finance for AAPL ...

🤖 AGENT RESPONSE:
The current stock price of Apple (AAPL) is 247.99.


In [ ]:
# 1. Define the new tool for Price-to-Earnings (P/E) ratio
def get_pe_ratio(ticker: str):
    """
    Fetches the trailing Price-to-Earnings (P/E) ratio for a given stock ticker.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL').
    """
    print(f"  ... 📊 TOOL CALL: Fetching P/E Ratio for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        # P/E ratio is typically found in the 'info' dictionary under 'trailingPE'.
        pe = stock.info.get('trailingPE', 'N/A')
        return {"pe_ratio": pe}
    except Exception as e:
        return f"Error fetching P/E ratio for {ticker}: {e}"


print("✅ New tool 'get_pe_ratio' defined.")

# 2. Re-initialize the Model with ALL financial tools
# It's crucial to include all desired tools when initializing the model.
tools_list = [get_stock_price, get_company_risk_score, get_pe_ratio]

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    tools=tools_list
)

# Start a fresh chat session, enabling automatic function calling.
# This allows the model to invoke the defined Python functions as needed.
chat = model.start_chat(enable_automatic_function_calling=True)
print("✅ Agent re-initialized with the new P/E ratio tool.")

# 3. Prompt the Agent to evaluate Apple's P/E ratio
query = "Fetch the P/E ratio for Apple. Tell me its current P/E, and then evaluate: is Apple 'overvalued' compared to the average market P/E of 25?"
print(f"\n❓ USER QUERY: {query}\n")

response = chat.send_message(query)

print("\n🤖 AGENT RESPONSE:")
print(response.text)

✅ New tool 'get_pe_ratio' defined.
✅ Agent re-initialized with the new P/E ratio tool.

❓ USER QUERY: Fetch the P/E ratio for Apple. Tell me its current P/E, and then evaluate: is Apple 'overvalued' compared to the average market P/E of 25?

  ... 📊 TOOL CALL: Fetching P/E Ratio for AAPL ...

🤖 AGENT RESPONSE:
Apple's current P/E ratio is 31.35. Compared to the average market P/E of 25, Apple appears to be overvalued as its P/E ratio is higher than the average.


# Insights & Technical Learnings

## Key Results
*   **Tool Integration Success**: The Gemini-2.5-Flash model successfully integrated and utilized custom tools (`get_stock_price`, `get_company_risk_score`, `get_pe_ratio`) to fetch real-time financial data.
*   **Contextual Understanding**: The agent demonstrated strong contextual understanding, correctly interpreting the user's query about P/E ratio and valuation, and then performing the necessary tool call to `get_pe_ratio`.
*   **Analytical Capability**: Beyond simple data retrieval, the agent was able to perform a basic comparative analysis (Apple's P/E vs. market average) and provide a valuation assessment, indicating its capability for more complex financial reasoning when guided.
*   **Efficiency**: The `yfinance` library, especially with `stock.fast_info`, proved efficient for rapid data retrieval, making the tool calls quick and responsive.
___
## Technical Learnings
*   **Agentic Workflow**: This task reinforced the power of agentic workflows where LLMs can act as intelligent orchestrators, deciding when and how to leverage external functions to extend their capabilities beyond their training data. The `enable_automatic_function_calling=True` feature in the `google-generativeai` SDK significantly streamlines this process.
*   **Tool Design**: The design of the `get_pe_ratio` tool illustrated the importance of robust error handling (using `try-except`) and returning clear, actionable information for the LLM to interpret.
*   **API Key Management**: Securely managing API keys via `google.colab.userdata` is a critical practice for public notebooks, preventing sensitive information from being exposed in code.
*   **Limitations of Pure LLM**: Without the `get_pe_ratio` tool, the LLM would be unable to provide real-time, accurate P/E ratios, potentially "hallucinating" outdated or incorrect financial data. This highlights the necessity of grounding LLMs with real-time data sources for financial applications.
___
## Practical Application
This intelligent agent architecture has significant practical applications in Fintech and AI:
*   **Personalized Financial Assistants**: Developing chatbots that can answer specific investment questions, provide real-time stock updates, and offer basic analytical insights to retail investors.
*   **Automated Investment Research**: Automating preliminary research for financial analysts by quickly gathering and synthesizing data points on thousands of stocks.
*   **Risk Assessment Tools**: Building dynamic tools that can assess and explain company risk profiles based on market metrics and news.
*   **Educational Platforms**: Creating interactive learning environments where users can query financial data and receive explanations in natural language.
*   **API Integration for Trading Bots**: The ability to programmatically fetch financial data can be a foundational component for more sophisticated automated trading strategies and portfolio management systems.